In [2]:
import pandas as pd 

df = pd.read_csv(r"/home/aboubakr/Desktop/enterprise-data-observability-platform/csv_denormalisation/bank_transactions_data_2_augmented_clean_2.csv")
df

,TransactionID,AccountID,TransactionAmount,TransactionDate,TransactionType,Location,DeviceID,IP Address,MerchantID,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance
0,TX000001,AC00128,14.09,4/11/2023 16:29,Debit,San Diego,D000380,162.198.218.92,M015,ATM,70,Doctor,81,1,5112.21
1,TX000002,AC00455,376.24,6/27/2023 16:44,Debit,Houston,D000051,13.149.61.4,M052,ATM,68,Doctor,141,1,13758.91
2,TX000003,AC00019,126.29,7/10/2023 18:16,Debit,Mesa,D000235,215.97.143.157,M009,Online,19,Student,56,1,1122.35
3,TX000004,AC00070,184.50,5/5/2023 16:32,Debit,Raleigh,D000187,200.13.225.150,M002,Online,26,Student,25,1,8569.06
4,TX000005,AC00411,13.45,10/16/2023 17:51,Credit,Atlanta,D000308,65.164.3.100,M091,Online,26,Student,198,1,7429.40
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,TX049996,AC00314,69.23,1/28/2025,Debit,Las Vegas,D000546,44.67.137.125,M097,Online,69,Doctor,69,1,6020.29
49996,TX049997,AC00370,514.53,1/23/2022,Debit,Houston,D000589,140.212.253.222,M061,ATM,46,Engineer,143,1,6371.51
49997,TX049998,AC00277,118.39,11/8/2022,Debit,Omaha,D000217,152.140.239.181,M029,Online,33,Doctor,296,1,749.34
49998,TX049999,AC00007,446.99,4/20/2025,Debit,Las Vegas,D000327,131.41.45.13,M082,ATM,58,Doctor,11,1,10915.11


In [7]:
df['TransactionDate'].dtype

<StringDtype(storage='python', na_value=nan)>

In [17]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# 1. Chargement des données
print("Chargement des données...")
df = pd.read_csv(r'/home/aboubakr/Desktop/enterprise-data-observability-platform/csv_denormalisation/bank_transactions_data_2_augmented_clean_2.csv')

# 2. Tri chronologique OBLIGATOIRE (pour les calculs de vélocité)
print("Traitement temporel...")
df['TransactionDate'] = pd.to_datetime(
    df['TransactionDate'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
)

# On supprime les lignes dont la date n'a pas pu être parsée
df = df.dropna(subset=['TransactionDate']).copy()
df = df.sort_values('TransactionDate').reset_index(drop=True)

# Extraction de la saisonnalité
df['tx_hour'] = df['TransactionDate'].dt.hour
df['tx_day_of_week'] = df['TransactionDate'].dt.dayofweek

# 3. Calcul des variables de Vélocité (Comportement)
print("Calcul des variables de vélocité...")

# On calcule les rolling features par groupe en gardant l'index d'origine
# pour éviter les problèmes d'alignement avec des dates dupliquées.
def compute_rolling_feature(group: pd.DataFrame, window: str, column: str, agg: str) -> pd.Series:
    group = group.sort_values('TransactionDate')
    if agg == 'count':
        values = group.rolling(window, on='TransactionDate')[column].count()
    elif agg == 'mean':
        values = group.rolling(window, on='TransactionDate')[column].mean()
    else:
        raise ValueError(f"Unsupported aggregation: {agg}")
    values.index = group.index
    return values

# Compte: Nombre de transactions sur 24h
df['account_tx_count_24h'] = (
    df.groupby('AccountID', group_keys=False)
      .apply(lambda group: compute_rolling_feature(group, '1d', 'TransactionID', 'count'))
      .sort_index()
)

# Compte: Montant moyen sur 7 jours
df['account_avg_amount_7d'] = (
    df.groupby('AccountID', group_keys=False)
      .apply(lambda group: compute_rolling_feature(group, '7d', 'TransactionAmount', 'mean'))
      .sort_index()
)

# Marchand: Nombre de transactions reçues dans la dernière heure
df['merchant_tx_count_1h'] = (
    df.groupby('MerchantID', group_keys=False)
      .apply(lambda group: compute_rolling_feature(group, '1h', 'TransactionID', 'count'))
      .sort_index()
)

# 4. Calcul des Ratios
# On ajoute 0.1 pour éviter la division par zéro si la moyenne est de 0
df['amount_vs_avg_ratio'] = df['TransactionAmount'] / (df['account_avg_amount_7d'] + 0.1)

# 5. Nettoyage et passage en index sécurisé
# TransactionID reste l'identifiant de jointure dans les deux CSV, sans entrer dans les features.
print("Configuration des clés primaires en index...")
metadata_df = df[['TransactionID', 'AccountID', 'DeviceID', 'IP Address', 'MerchantID', 'TransactionDate']].copy()
metadata_df = metadata_df.set_index('TransactionID')

df_ml = df.drop(columns=['AccountID', 'DeviceID', 'IP Address', 'MerchantID', 'TransactionDate']).copy()
df_ml = df_ml.set_index('TransactionID')

# 6. One-Hot Encoding des variables catégorielles (Textes -> 0/1)
print("Encodage des catégories...")
colonnes_categorielles = ['TransactionType', 'Location', 'Channel', 'CustomerOccupation']
df_ml = pd.get_dummies(df_ml, columns=colonnes_categorielles, drop_first=True)

# 7. Normalisation Min-Max (Tout entre 0 et 1)
print("Normalisation des données...")
scaler = MinMaxScaler()
# On réinjecte l'index d'origine pour garder la correspondance ligne ↔ TransactionID
df_ml_scaled = pd.DataFrame(scaler.fit_transform(df_ml), columns=df_ml.columns, index=df_ml.index)

# 8. Sauvegarde du dataset prêt pour le Machine Learning
print("Sauvegarde des fichiers ML...")
df_ml_scaled.to_csv(r'/home/aboubakr/Desktop/enterprise-data-observability-platform/csv_denormalisation/ml_ready_transactions.csv', index=True)

# Sauvegarde des métadonnées pour faire la jointure après l'inférence
metadata_df.to_csv(r'/home/aboubakr/Desktop/enterprise-data-observability-platform/csv_denormalisation/ml_metadata.csv', index=True)

print(f"Terminé ! La base de données ML contient {df_ml_scaled.shape[1]} colonnes (features).")

Chargement des données...
Traitement temporel...
Calcul des variables de vélocité...


/tmp/ipykernel_99014/1475327134.py:34: Pandas4Warning: 'd' is deprecated and will be removed in a future version, please use 'D' instead.
  values = group.rolling(window, on='TransactionDate')[column].count()
/tmp/ipykernel_99014/1475327134.py:34: Pandas4Warning: 'd' is deprecated and will be removed in a future version, please use 'D' instead.
  values = group.rolling(window, on='TransactionDate')[column].count()
/tmp/ipykernel_99014/1475327134.py:34: Pandas4Warning: 'd' is deprecated and will be removed in a future version, please use 'D' instead.
  values = group.rolling(window, on='TransactionDate')[column].count()
/tmp/ipykernel_99014/1475327134.py:34: Pandas4Warning: 'd' is deprecated and will be removed in a future version, please use 'D' instead.
  values = group.rolling(window, on='TransactionDate')[column].count()
/tmp/ipykernel_99014/1475327134.py:34: Pandas4Warning: 'd' is deprecated and will be removed in a future version, please use 'D' instead.
  values = group.rolling(w

Configuration des clés primaires en index...
Encodage des catégories...
Normalisation des données...
Sauvegarde des fichiers ML...
Terminé ! La base de données ML contient 59 colonnes (features).
